In [3]:
import re, csv
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = Path("../data")
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

In [4]:
tox = pd.read_csv(DATA / "raw" / "train.csv")
print(tox.shape)


(159571, 8)


In [5]:
def clean_text(t):
    t = str(t).lower()                              # same word, same case
    t = re.sub(r"http\S+|www\.\S+", " ", t)         # remove links
    t = re.sub(r"\d{1,3}(\.\d{1,3}){3}", " ", t)   # remove IP addresses
    t = re.sub(r"@\w+", " ", t)                     # remove @mentions
    t = re.sub(r"(.)\1{3,}", r"\1\1\1", t)          # stuuuuupid -> stuuupid
    return re.sub(r"\s+", " ", t).strip()           # fix spaces

tox["clean_text"] = tox["comment_text"].apply(clean_text)
tox = tox[tox["clean_text"] != ""].drop_duplicates("clean_text").reset_index(drop=True)
tox["any_toxic"] = (tox[LABELS].sum(axis=1) > 0).astype(int)
print(tox.shape)

(159189, 10)


In [6]:
def split_save(df, col, folder):
    train, rest = train_test_split(df, test_size=0.30, random_state=42, stratify=df[col])
    val, test = train_test_split(rest, test_size=0.50, random_state=42, stratify=rest[col])
    out = DATA / "processed" / folder
    out.mkdir(parents=True, exist_ok=True)
    for name, part in [("train", train), ("val", val), ("test", test)]:
        part.to_csv(out / f"{name}.csv", index=False)
        print(f"{folder:9} {name:5} {len(part):>7} rows | {part[col].mean() * 100:.1f}% positive")

split_save(tox, "any_toxic", "toxicity")

toxicity  train  111432 rows | 10.1% positive
toxicity  val     23878 rows | 10.1% positive
toxicity  test    23879 rows | 10.1% positive


In [8]:
sms = pd.read_csv(DATA / "raw" / "SMSSpamCollection", sep="\t", header=None,
                  names=["label", "text"], quoting=csv.QUOTE_NONE)
sms["spam"] = (sms["label"] == "spam").astype(int)

def clean_spam(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " urltoken ", t)   # keep "there was a link"
    t = re.sub(r"\d{5,}", " longnumber ", t)           # keep "there was a long number"
    return re.sub(r"\s+", " ", t).strip()

sms["clean_text"] = sms["text"].apply(clean_spam)
sms = sms.drop_duplicates("clean_text").reset_index(drop=True)
print(sms["label"].value_counts())

split_save(sms, "spam", "spam")

label
ham     4517
spam     626
Name: count, dtype: int64
spam      train    3600 rows | 12.2% positive
spam      val       771 rows | 12.2% positive
spam      test      772 rows | 12.2% positive
